In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context('paper', font_scale=1.4)
sns.set_style('whitegrid')
os.makedirs('out', exist_ok=True)

In [ ]:
COLORDICT = {
    'scDEF': '#2ca02c',
    'scDEF_un': '#98df8a',
    'scDEF_corr': '#17becf',
    'scDEF_hclust': '#9edae5',
    'PCA': '#9467bd',
    'NMF': '#1f77b4',
    'scHPF': '#aec7e8',
    'scVI': '#ff7f0e',
    'Harmony': '#d62728',
    'Scanorama': '#e377c2',
    'nSBM': '#8c564b',
    'fscLVM': '#bcbd22',
}

METRICS_SINGLEBATCH = [
    'Cell Type ARI', 'Cell Type ASW',
    'Hierarchy accuracy', 'Hierarchical signature consistency',
    'Signature accuracy', 'Signature sparsity',
]

METRICS_MULTIBATCH = [
    'Cell Type ARI', 'Cell Type ASW',
    'Batch ARI', 'Batch ASW',
    'Hierarchy accuracy', 'Hierarchical signature consistency',
    'Signature accuracy', 'Signature sparsity',
]

RESULTS_DIR = '..'

In [ ]:
def boxplot_metrics(df, metrics, group_col=None, figsize=None, title=None, palette=None):
    """Box-plot of each metric across methods, optionally faceted by a grouping column."""
    if palette is None:
        palette = COLORDICT
    metrics_in_data = [m for m in metrics if m in df['score'].unique()]
    sub = df[df['score'].isin(metrics_in_data)].copy()
    sub['value'] = sub['value'].astype(float)

    if group_col is not None:
        groups = sorted(sub[group_col].unique())
        n_groups = len(groups)
        n_metrics = len(metrics_in_data)
        if figsize is None:
            figsize = (5 * n_metrics, 4 * n_groups)
        fig, axes = plt.subplots(n_groups, n_metrics, figsize=figsize,
                                 squeeze=False, sharey=False)
        for gi, g in enumerate(groups):
            gdf = sub[sub[group_col] == g]
            for mi, metric in enumerate(metrics_in_data):
                ax = axes[gi, mi]
                mdf = gdf[gdf['score'] == metric]
                sns.boxplot(data=mdf, x='method', y='value', palette=palette,
                            order=[m for m in palette if m in mdf['method'].unique()],
                            ax=ax, showfliers=False)
                sns.stripplot(data=mdf, x='method', y='value', color='black',
                              size=3, alpha=0.5,
                              order=[m for m in palette if m in mdf['method'].unique()],
                              ax=ax)
                ax.set_title(f'{group_col}={g}', fontsize=11)
                ax.set_ylabel(metric if gi == 0 else '')
                ax.set_xlabel('')
                ax.tick_params(axis='x', rotation=45)
    else:
        n_metrics = len(metrics_in_data)
        if figsize is None:
            figsize = (5 * n_metrics, 5)
        fig, axes = plt.subplots(1, n_metrics, figsize=figsize, squeeze=False)
        for mi, metric in enumerate(metrics_in_data):
            ax = axes[0, mi]
            mdf = sub[sub['score'] == metric]
            sns.boxplot(data=mdf, x='method', y='value', palette=palette,
                        order=[m for m in palette if m in mdf['method'].unique()],
                        ax=ax, showfliers=False)
            sns.stripplot(data=mdf, x='method', y='value', color='black',
                          size=3, alpha=0.5,
                          order=[m for m in palette if m in mdf['method'].unique()],
                          ax=ax)
            ax.set_ylabel(metric)
            ax.set_xlabel('')
            ax.tick_params(axis='x', rotation=45)

    if title:
        fig.suptitle(title, fontsize=16, y=1.02)
    plt.tight_layout()
    return fig


def lineplot_sweep(df, sweep_col, metrics, hue=None, figsize=None, title=None):
    """Line plot showing how metrics change across a swept parameter."""
    metrics_in_data = [m for m in metrics if m in df['score'].unique()]
    sub = df[df['score'].isin(metrics_in_data)].copy()
    sub['value'] = sub['value'].astype(float)
    sub[sweep_col] = sub[sweep_col].astype(float)

    n_metrics = len(metrics_in_data)
    if figsize is None:
        figsize = (5 * n_metrics, 4)
    fig, axes = plt.subplots(1, n_metrics, figsize=figsize, squeeze=False)
    for mi, metric in enumerate(metrics_in_data):
        ax = axes[0, mi]
        mdf = sub[sub['score'] == metric]
        sns.lineplot(data=mdf, x=sweep_col, y='value', hue=hue,
                     marker='o', ax=ax, err_style='bars', errorbar='sd')
        ax.set_ylabel(metric)
        ax.set_xlabel(sweep_col)
    if title:
        fig.suptitle(title, fontsize=16, y=1.02)
    plt.tight_layout()
    return fig


def walltime_plot(df, figsize=(10, 6)):
    """Log-log plot of runtime vs number of cells for each method."""
    sub = df[df['score'] == 'Runtime'].copy()
    sub['value'] = sub['value'].astype(float)
    sub['cellno'] = sub['cellno'].astype(int)
    fig, ax = plt.subplots(figsize=figsize)
    for method in sub['method'].unique():
        mdf = sub[sub['method'] == method].sort_values('cellno')
        color = COLORDICT.get(method, 'gray')
        ax.plot(mdf['cellno'], mdf['value'], marker='o', label=method, color=color)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Number of cells')
    ax.set_ylabel('Runtime (s)')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    ax.grid(True, which='both', alpha=0.3)
    plt.tight_layout()
    return fig

## 1. Single-batch benchmark

In [ ]:
sb = pd.read_csv(f'{RESULTS_DIR}/results/benchmark_singlebatch/singlebatch_scores.csv')
sb.head()

In [ ]:
fig = boxplot_metrics(sb, METRICS_SINGLEBATCH, group_col='separability',
                      title='Single-batch benchmark')
fig.savefig('out/singlebatch_benchmark.pdf', bbox_inches='tight')

## 2. Multi-batch benchmark

In [ ]:
mb = pd.read_csv(f'{RESULTS_DIR}/results/benchmark_multibatch/multibatch_scores.csv')
mb.head()

In [ ]:
fig = boxplot_metrics(mb, METRICS_MULTIBATCH, group_col='separability',
                      title='Multi-batch benchmark (all frac_shared)')
fig.savefig('out/multibatch_benchmark.pdf', bbox_inches='tight')

## 3. Hyperparameter ablations

In [ ]:
try:
    hw = pd.read_csv(f'{RESULTS_DIR}/results/hyperparams/hierarchy_weight_scores.csv')
    brd_s = pd.read_csv(f'{RESULTS_DIR}/results/hyperparams/brd_strength_scores.csv')
    brd_m = pd.read_csv(f'{RESULTS_DIR}/results/hyperparams/brd_mean_scores.csv')
except FileNotFoundError as e:
    print(f"Hyperparameter results not found: {e}")
    hw = brd_s = brd_m = None

In [ ]:
ablation_metrics = [
    'Cell Type ARI', 'Hierarchy accuracy',
    'Hierarchical signature consistency', 'Signature accuracy',
]

if hw is not None:
    fig = lineplot_sweep(hw, 'hierarchy_weight', ablation_metrics,
                         hue='density', title='Hierarchy weight sweep')
    fig.savefig('out/hw_sweep.pdf', bbox_inches='tight')
else:
    print("Skipping: hierarchy_weight results not available")

In [ ]:
if brd_s is not None:
    fig = lineplot_sweep(brd_s, 'brd_strength', ablation_metrics,
                         hue='density', title='BRD strength sweep')
    fig.savefig('out/brd_strength_sweep.pdf', bbox_inches='tight')
else:
    print("Skipping: brd_strength results not available")

In [ ]:
if brd_m is not None:
    fig = lineplot_sweep(brd_m, 'brd_mean', ablation_metrics,
                         hue='density', title='BRD mean sweep')
    fig.savefig('out/brd_mean_sweep.pdf', bbox_inches='tight')
else:
    print("Skipping: brd_mean results not available")

## 4. Structure ablation

In [ ]:
try:
    layers = pd.read_csv(f'{RESULTS_DIR}/results/structure/layers_scores.csv')
    factors = pd.read_csv(f'{RESULTS_DIR}/results/structure/factors_scores.csv')
except FileNotFoundError as e:
    print(f"Structure results not found: {e}")
    layers = factors = None

In [ ]:
if layers is not None:
    fig = lineplot_sweep(layers, 'n_layers', ablation_metrics,
                         title='Number of layers sweep')
    fig.savefig('out/layers_sweep.pdf', bbox_inches='tight')
else:
    print("Skipping: layers results not available")

In [ ]:
if factors is not None:
    fig = lineplot_sweep(factors, 'n_factors', ablation_metrics,
                         title='Number of factors sweep')
    fig.savefig('out/factors_sweep.pdf', bbox_inches='tight')
else:
    print("Skipping: factors results not available")

## 5. Wall-time scalability

In [ ]:
try:
    wt = pd.read_csv(f'{RESULTS_DIR}/results/walltime/walltime_scores.csv')
    fig = walltime_plot(wt)
    fig.savefig('out/walltime.pdf', bbox_inches='tight')
except FileNotFoundError:
    print("Skipping: walltime results not available")